In [9]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, SVR
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, \
RocCurveDisplay, roc_auc_score, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import log_loss, f1_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor, BaggingClassifier, BaggingRegressor, RandomForestClassifier, RandomForestRegressor, StackingClassifier, StackingRegressor
from sklearn.linear_model import ridge_regression, ElasticNet
from sklearn.linear_model import Ridge


import xgboost as xgb
import lightgbm as lgb
import catboost

In [2]:
df = pd.read_csv('../Cases/Concrete_Strength/Concrete_Data.csv')
df

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30
...,...,...,...,...,...,...,...,...,...
1025,276.4,116.0,90.3,179.6,8.9,870.1,768.3,28,44.28
1026,322.2,0.0,115.6,196.0,10.4,817.9,813.4,28,31.18
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28,23.70
1028,159.1,186.7,0.0,175.6,11.3,989.6,788.9,28,32.77


In [ ]:
x,y = df.drop('Strength', axis = 1),df['Strength']


Scaling

In [ ]:
scaler = StandardScaler()

x = scaler.fit_transform(x)
x_train, x_test , y_train, y_test = train_test_split(x,y, random_state=25, test_size=0.3)

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [19]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


xgb = XGBRegressor(random_state = 25, verbose = -1)
cb = catboost.CatBoostRegressor(random_state=25, verbose = 0)
lgb = LGBMRegressor(random_state = 25, verbose = -1)
rf = RandomForestRegressor(random_state=25)


lst_final_estimators = [rf, xgb, cb,lgb]

passthorugh_lst = [True, False]

lr = LinearRegression()
ridge = Ridge()
lasso = Lasso()
elastic = ElasticNet()
dtc = DecisionTreeRegressor(random_state=25)

base_models = [('LR', lr),('Ridge', ridge),('Tree', dtc),('Lassso',lasso),('Elastic Net',elastic)]

scores = []
for final_estimator in tqdm(lst_final_estimators):
    for p in passthorugh_lst:
        stack = StackingRegressor(estimators = base_models,
                              final_estimator=final_estimator,
                              passthrough=p)

        stack.fit(x_train, y_train)
        y_pred = stack.predict(x_test)

        scores.append([final_estimator, p, mean_absolute_error(y_test, y_pred)])

df_scores = pd.DataFrame(data = scores, columns=['Final Estimators', 'Passthrough', 'mean_absolute_error'])
df_scores.sort_values('mean_absolute_error', ascending=True)

100%|██████████| 4/4 [00:07<00:00,  1.89s/it]


,Final Estimators,Passthrough,mean_absolute_error
4,<catboost.core.CatBoostRegressor object at 0x0...,True,3.212097
6,"LGBMRegressor(random_state=25, verbose=-1)",True,3.568845
2,"XGBRegressor(base_score=None, booster=None, ca...",True,3.728863
0,RandomForestRegressor(random_state=25),True,3.986554
5,<catboost.core.CatBoostRegressor object at 0x0...,False,4.361621
7,"LGBMRegressor(random_state=25, verbose=-1)",False,4.615894
1,RandomForestRegressor(random_state=25),False,4.645566
3,"XGBRegressor(base_score=None, booster=None, ca...",False,5.152736


# using r2score as a metric

In [20]:
scores = []
for final_estimator in tqdm(lst_final_estimators):
    for p in passthorugh_lst:
        stack = StackingRegressor(estimators = base_models,
                              final_estimator=final_estimator,
                              passthrough=p)

        stack.fit(x_train, y_train)
        y_pred = stack.predict(x_test)

        scores.append([final_estimator, p, r2_score(y_test, y_pred)])

df_scores = pd.DataFrame(data = scores, columns=['Final Estimators', 'Passthrough', 'R2score'])
df_scores.sort_values('R2score', ascending=False)

100%|██████████| 4/4 [00:07<00:00,  1.78s/it]


,Final Estimators,Passthrough,R2score
4,<catboost.core.CatBoostRegressor object at 0x0...,True,0.905984
6,"LGBMRegressor(random_state=25, verbose=-1)",True,0.896125
2,"XGBRegressor(base_score=None, booster=None, ca...",True,0.883402
0,RandomForestRegressor(random_state=25),True,0.875477
5,<catboost.core.CatBoostRegressor object at 0x0...,False,0.851392
1,RandomForestRegressor(random_state=25),False,0.848963
7,"LGBMRegressor(random_state=25, verbose=-1)",False,0.844012
3,"XGBRegressor(base_score=None, booster=None, ca...",False,0.817791
